# Workspace RayCluster + job client — distributed compute

**Ray library this topic:** [Ray Core](https://docs.ray.io/en/latest/ray-core/walkthrough.html) (`@ray.remote` tasks).

**Same platform as Topics 1 and 3:** shared `ray-workshop` cluster + `job_client`.

**Different from Topic 1** (Ray Data pipeline) and **Topic 3** (Ray Train + MLflow): this script fans out plain Python remote functions — no Dataset API, no Train, no GPUs required (cluster still has GPUs so Topics 1–3 share one shape).

Runs `distributed_stats.py`. Leave the cluster up for Topic 3.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 1 — OpenShift Console → **Copy login command** → Display token.


In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## Create or reuse the RayCluster

Same shared **`ray-workshop`** shape as Topic 1. `apply()` creates if missing, updates if already running.


In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

# Shared name across Topics 1–3. apply() creates if missing, updates if present.
cluster = Cluster(
    ClusterConfiguration(
        name="ray-workshop",
        namespace=NAMESPACE,
        local_queue=LOCAL_QUEUE,
        num_workers=2,
        head_cpu_requests=5,
        head_cpu_limits=8,
        head_memory_requests=4,
        head_memory_limits=8,
        worker_cpu_requests=1,
        worker_cpu_limits=2,
        worker_extended_resource_requests={"nvidia.com/gpu": 1},
        worker_memory_requests=4,
        worker_memory_limits=6,
        write_to_file=False,
    )
)
cluster.apply()
cluster.wait_ready()
cluster.details()


## Submit with `job_client`

Same submit pattern as Topic 1: a Ray Job on the existing cluster. `runtime_env.working_dir` ships the repo; `env_vars` set `INPUT_PATH` and `NUM_PARTITIONS`.

**What Ray Core does in `distributed_stats.py`:**

1. Load Iris CSV and split rows into partitions
2. Launch `@ray.remote` `summarize_partition` tasks (one per partition) on workers
3. `ray.get` the results and print aggregated row counts

Expect partition JSON and `Aggregated row count: 30` in the logs.


In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/distributed_stats.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "NUM_PARTITIONS": "4",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 900

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))

In [ ]:
# Optional — skip to keep the cluster for Topic 3
# cluster.down()
# print("Cluster down.")
print("Leaving cluster up for Topic 3. Uncomment cluster.down() above if needed.")


## Check results

Expect JSON partition summaries and `Aggregated row count: 30` in the job logs above.

Optional CLI (while the cluster is up):

```sh
oc logs -n ray-workshop -l ray.io/cluster=ray-workshop,ray.io/node-type=head -c ray-head --tail=100
```
